In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel('d:/hcdcarbody/hcdcarbody/simulation/data/车身库数据v20260115.xlsx')
df

,任务号,类型,源巷道,源货位,目标巷道,目标货位,RFID,创建时间,开始时间,完成时间,执行耗时（s)
0,82491,端口1入库,NaN,NaN,3.0,6-1-5,HJ4ABBHK1TN110032,2026-01-14 09:45:42.560,2026-01-14 09:45:44.577,2026-01-14 09:48:11.990,147.413002
1,82483,端口2出库,2.0,3-2-5,NaN,NaN,HJ4BACJH1TN109989,2026-01-14 09:32:55.907,2026-01-14 09:46:00.733,2026-01-14 09:47:35.257,94.524002
2,82490,端口1入库,NaN,NaN,2.0,3-3-5,HJ4BACDH7TN110012,2026-01-14 09:43:43.233,2026-01-14 09:43:45.050,2026-01-14 09:45:58.073,133.023004
3,82489,端口1入库,NaN,NaN,1.0,2-3-5,HJ4ABBBE0TN110171,2026-01-14 09:42:09.013,2026-01-14 09:42:11.093,2026-01-14 09:44:37.047,145.953999
4,82482,端口2出库,3.0,5-4-5,NaN,NaN,HJ4BACJHXTN109991,2026-01-14 09:32:55.893,2026-01-14 09:42:52.520,2026-01-14 09:44:26.203,93.683002
...,...,...,...,...,...,...,...,...,...,...,...
81606,20,移库,1.0,1-1-3,1.0,2-1-3,111111111111111112222222222222222223333,2025-06-05 08:57:26.277,2025-06-05 08:57:27.643,2025-06-05 08:58:09.647,42.003999
81607,19,移库,1.0,1-1-1,1.0,1-1-3,111111111111111112222222222222222223333,2025-06-05 08:56:22.453,2025-06-05 08:56:23.420,2025-06-05 08:57:26.183,62.763008
81608,18,移库,1.0,1-1-2,1.0,1-1-1,111111111111111112222222222222222223333,2025-06-05 08:55:26.723,2025-06-05 08:55:28.083,2025-06-05 08:56:21.900,53.816996
81609,17,移库,1.0,1-1-1,1.0,1-1-2,111111111111111112222222222222222223333,2025-06-05 08:10:39.783,2025-06-05 08:54:32.253,2025-06-05 08:55:26.577,54.324000


In [38]:
def extract_matched_tasks_with_separate_inbound(df: pd.DataFrame):
    df = df.copy()

    # 1) 只保留入库/出库任务（支持 端口1入库 / 端口2出库 这种形式）
    df = df[df["类型"].astype(str).str.contains("入库|出库", na=False)].copy()

    # 2) 跳过 RFID 包含 KHQ 的行
    df = df[~df["RFID"].astype(str).str.contains("KHQ", na=False)].copy()

    # 3) 时间列转 datetime，删除时间为空的行
    for col in ["创建时间", "开始时间", "完成时间"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    df = df[df["创建时间"].notna() & df["开始时间"].notna() & df["完成时间"].notna()]

    # 4) 标记入库 / 出库（依然基于“是否包含入库/出库”）
    is_in = df["类型"].astype(str).str.contains("入库", na=False)
    is_out = df["类型"].astype(str).str.contains("出库", na=False)

    in_df = df[is_in].copy()
    out_df = df[is_out].copy()

    # 5) 对每条出库任务，找对应的入库任务
    matched_tasks = []
    used_in_rfid = set()
    matched_inbound_records = []

    in_groups = {k: g.sort_values("完成时间") for k, g in in_df.groupby("RFID")}

    for rfid, g_out in out_df.groupby("RFID"):
        if rfid not in in_groups:
            continue

        g_in = in_groups[rfid]
        in_times = g_in["完成时间"].to_numpy()

        for idx, out_row in g_out.iterrows():
            t = out_row["开始时间"]
            pos = np.searchsorted(in_times, np.datetime64(t), side="right") - 1
            if pos < 0:
                continue

            in_row = g_in.iloc[pos]

            matched_tasks.append((out_row, in_row))
            matched_inbound_records.append(in_row.copy())  # ✅ 保留完整原始入库行
            used_in_rfid.add(in_row["RFID"])

    # 6) 将配对信息回写到出库任务
    matched_df = []
    for out_row, in_row in matched_tasks:
        matched_record = out_row.copy()
        matched_record["matched_in_task_id"] = in_row["任务号"]
        matched_record["matched_in_aisle"] = in_row["目标巷道"]
        matched_record["matched_in_pos"] = in_row["目标货位"]
        matched_record["matched_in_create_time"] = in_row["创建时间"]
        matched_record["matched_in_start_time"] = in_row["开始时间"]
        matched_record["matched_in_finish_time"] = in_row["完成时间"]
        matched_record["matched_in_duration_s"] = in_row["执行耗时（s)"]
        matched_record["matched_in_type"] = in_row["类型"]  # ✅ 新增：保留端口入库类型

        matched_df.append(matched_record)

    matched_df = pd.DataFrame(matched_df)

    # 7) 已使用的入库任务（完整原始结构，含“端口1入库”）
    matched_inbound_df = pd.DataFrame(matched_inbound_records)

    # 8) 剩余未使用入库任务
    in_df_remaining = in_df[~in_df["RFID"].isin(used_in_rfid)]

    # 9) 返回三个结果
    return matched_df, matched_inbound_df, in_df_remaining

In [39]:
df1,df2,df3 = extract_matched_tasks_with_separate_inbound(df)

In [42]:
def build_used_in_out_from_matched(
    df1: pd.DataFrame,
    start_time: str | pd.Timestamp = "2026-01-01",
    *,
    skip_khq: bool = True,
    require_matched_inbound: bool = True,
    sort_result: bool = True,
) -> pd.DataFrame:
    """
    仅使用 matched_df（df1）构造“已使用的出库任务 + 已使用的入库任务”拼接表（只保留初始特征）。

    适用前提：
    - df1 是你已经完成 RFID 匹配后的出库表（每行是一条出库任务），
      且包含 matched_in_* 的入库信息列（如 matched_in_task_id 等）。
    - 你希望按出库开始时间筛选，并将匹配到的入库任务与出库任务上下拼接到同一张表中。
    - RFID 中包含 "KHQ" 的先跳过（可选）。

    返回：
    - used_all_df：包含入库+出库两类任务，字段为初始特征列 + _role（inbound/outbound）
    """

    df = df1.copy()

    # -------- 1) 时间列安全转换 --------
    # 出库原始时间列
    for col in ["创建时间", "开始时间", "完成时间"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # 入库匹配时间列
    inbound_time_cols = [
        "matched_in_create_time",
        "matched_in_start_time",
        "matched_in_finish_time",
    ]
    for col in inbound_time_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # -------- 2) 按出库开始时间筛选 --------
    ts = pd.Timestamp(start_time)
    if "开始时间" not in df.columns:
        raise KeyError("df1 中缺少“开始时间”列，无法按出库开始时间筛选。")

    df_filter = df[df["开始时间"].notna() & (df["开始时间"] >= ts)].copy()

    # 可选：跳过 KHQ
    if skip_khq and "RFID" in df_filter.columns:
        df_filter = df_filter[~df_filter["RFID"].astype(str).str.contains("KHQ", na=False)].copy()

    # 可选：只保留确实匹配到入库的出库行
    if require_matched_inbound:
        if "matched_in_task_id" not in df_filter.columns:
            raise KeyError("df1 中缺少 matched_in_task_id 列，无法判断是否匹配到入库。")
        df_filter = df_filter[df_filter["matched_in_task_id"].notna()].copy()

    # -------- 3) 定义“初始特征列”（不改字段名）--------
    base_cols = [
        "任务号", "类型",
        "源巷道", "源货位",
        "目标巷道", "目标货位",
        "RFID",
        "创建时间", "开始时间", "完成时间",
        "执行耗时（s)"
    ]

    # 检查缺失列（如果你的表确实没有某些列，可在这里删掉或改成 reindex）
    missing = [c for c in base_cols if c not in df_filter.columns]
    if missing:
        # 使用 reindex 容错：缺失列自动补 NaN（更稳）
        out_used = df_filter.reindex(columns=base_cols).copy()
    else:
        out_used = df_filter.loc[:, base_cols].copy()

    out_used["_role"] = "outbound"

    # -------- 4) 从 df_filter 的 matched_in_* 列构造“已使用入库任务表”--------
    # 这些列名来自你前面匹配时写入的字段
    required_in_cols = [
        "matched_in_task_id",
        "matched_in_type",
        "matched_in_aisle",
        "matched_in_pos",
        "matched_in_create_time",
        "matched_in_start_time",
        "matched_in_finish_time",
        "matched_in_duration_s",
        "RFID",
    ]
    missing_in = [c for c in required_in_cols if c not in df_filter.columns]
    if missing_in:
        raise KeyError(f"df1 中缺少以下入库匹配列，无法从 df1 直接构造入库任务：{missing_in}")

    in_used = pd.DataFrame({
        "任务号": df_filter["matched_in_task_id"],
        "类型": df_filter["matched_in_type"],
        "源巷道": np.nan,
        "源货位": np.nan,
        "目标巷道": df_filter["matched_in_aisle"],
        "目标货位": df_filter["matched_in_pos"],
        "RFID": df_filter["RFID"],
        "创建时间": df_filter["matched_in_create_time"],
        "开始时间": df_filter["matched_in_start_time"],
        "完成时间": df_filter["matched_in_finish_time"],
        "执行耗时（s)": df_filter["matched_in_duration_s"],
    })
    in_used["_role"] = "inbound"

    # -------- 5) 拼接 --------
    used_all_df = pd.concat([out_used, in_used], ignore_index=True)

    # -------- 6) 可选：排序（按 RFID + 创建时间）--------
    if sort_result:
        sort_keys = []
        if "RFID" in used_all_df.columns:
            sort_keys.append("RFID")
        if "创建时间" in used_all_df.columns:
            sort_keys.append("创建时间")
        if sort_keys:
            used_all_df = used_all_df.sort_values(sort_keys, kind="mergesort").reset_index(drop=True)

    # -------- 7) 可选：一致性检查（可注释掉）--------
    # 出库与入库应数量一致（如果 require_matched_inbound=True）
    if require_matched_inbound:
        vc = used_all_df["_role"].value_counts(dropna=False)
        if vc.get("outbound", 0) != vc.get("inbound", 0):
            # 不抛异常，只给出提示信息（按需可改成 raise）
            print(f"[WARN] outbound({vc.get('outbound', 0)}) != inbound({vc.get('inbound', 0)}), 请检查匹配/筛选条件。")

    return used_all_df


# ---------------------------
# 使用示例
# ---------------------------
# df1 = ...  # 你的 matched_df（包含 matched_in_* 列）
# used_all_df = build_used_in_out_from_matched(df1, start_time="2026-01-01")
# used_all_df.to_excel("used_in_out.xlsx", index=False)


In [43]:
used_all_df = build_used_in_out_from_matched(df1, start_time="2026-01-01")

In [45]:
used_all_df.sort_values(["完成时间"])

,任务号,类型,源巷道,源货位,目标巷道,目标货位,RFID,创建时间,开始时间,完成时间,执行耗时（s),_role
132,77097,端口1入库,NaN,NaN,1.0,1-1-3,HJ4ABB3H6TN107655,2025-12-26 16:36:08.243,2025-12-26 16:36:10.033,2025-12-26 16:38:49.387,159.354000,inbound
162,77101,端口1入库,NaN,NaN,1.0,1-1-4,HJ4ABB3H8TN107656,2025-12-26 16:41:49.627,2025-12-26 16:41:51.340,2025-12-26 16:44:31.203,159.863008,inbound
188,77108,端口1入库,NaN,NaN,1.0,1-1-1,HJ4ABB3HXTN107657,2025-12-26 16:49:24.623,2025-12-26 16:49:26.420,2025-12-26 16:52:06.370,159.949996,inbound
46,77122,端口1入库,NaN,NaN,3.0,5-3-5,HJ4ABB3H1TN107658,2025-12-26 17:11:53.370,2025-12-26 17:11:55.860,2025-12-26 17:13:59.403,123.542998,inbound
82,77132,端口1入库,NaN,NaN,1.0,1-4-3,HJ4ABB3H3TN107659,2025-12-26 17:21:56.660,2025-12-26 17:22:51.237,2025-12-26 17:25:19.797,148.560005,inbound
...,...,...,...,...,...,...,...,...,...,...,...,...
4409,82479,端口2出库,3.0,6-1-4,NaN,NaN,HJ4BACJHXTN109988,2026-01-14 09:32:55.847,2026-01-14 09:37:58.087,2026-01-14 09:39:28.227,90.139997,outbound
3941,82480,端口2出库,1.0,2-6-2,NaN,NaN,HJ4BACJH5TN109994,2026-01-14 09:32:55.857,2026-01-14 09:39:42.753,2026-01-14 09:41:43.747,120.993998,outbound
3853,82481,端口2出库,2.0,4-3-5,NaN,NaN,HJ4BACJH4TN109985,2026-01-14 09:32:55.877,2026-01-14 09:41:25.060,2026-01-14 09:42:59.227,94.166997,outbound
4411,82482,端口2出库,3.0,5-4-5,NaN,NaN,HJ4BACJHXTN109991,2026-01-14 09:32:55.893,2026-01-14 09:42:52.520,2026-01-14 09:44:26.203,93.683002,outbound


In [47]:
used_all_df.sort_values(["完成时间"]).to_excel("d:/hcdcarbody/hcdcarbody/simulation/data/used_data.xlsx")